## BASE

In [ ]:
import os
os.environ["XFORMERS_DISABLED"] = "1"
import gc, re, time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm
import sys; sys.path.append("/lustre/.../GradientDistillation")

In [ ]:
# DATA_ROOT = "/home/alex/internship/datasets/aqua20/data/aqua20"
# DATA_ROOT = "/Users/alex/Developpement/Internship/datasets/aqua20/data/aqua20"
DATA_ROOT = "/lustre/fswork/projects/rech/rbw/ucw75ke/datasets/aqua20/data/aqua20"
NUM_CLASSES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
MOCOV3_CKPT = "/lustre/fswork/projects/rech/rbw/ucw75ke/weights/mocov3_vit-b-300ep.pth.tar"

# stats utilisées À LA DISTILLATION (DINOv2 = ImageNet) -> espace des images synthétiques
_SRC_MEAN = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
_SRC_STD  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
PROBE_SEED = 0

In [ ]:
ARCHS = {
    "dinov2_vitb": dict(res=RESOLUTION, mean=IMAGENET_MEAN, std=IMAGENET_STD),
    "clip_vitb":   dict(res=224, mean=CLIP_MEAN,     std=CLIP_STD),
    "mocov3_vitb": dict(res=224, mean=IMAGENET_MEAN, std=IMAGENET_STD),
}

In [ ]:
@dataclass
class Backbone:
    name: str
    model: nn.Module
    forward: Callable[[torch.Tensor], torch.Tensor]
    mean: tuple
    std: tuple
    res: int
    feat_dim: int

In [ ]:
def load_backbone(name: str) -> Backbone:
    cfg = ARCHS[name]
    if name == "dinov2_vitb":
        m = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
        fwd = lambda x: m(x)                                   # CLS, 768
    elif name == "clip_vitb":
        import clip
        m = clip.load("ViT-B/32")[0].visual.float()
        fwd = lambda x: m(x)                                   # 512
    elif name == "mocov3_vitb":
        from src.models.moco_vision_tansformer import VisionTransformerMoCoV3
        m = VisionTransformerMoCoV3.from_pretrained("nyu-visionx/moco-v3-vit-b", num_classes=0)
        fwd = lambda x: m(x)                                   # 768
    else:
        raise ValueError(name)
    m.eval().requires_grad_(False).to(DEVICE)
    with torch.no_grad():
        feat_dim = fwd(torch.zeros(1, 3, cfg["res"], cfg["res"], device=DEVICE)).shape[-1]
    return Backbone(name, m, fwd, cfg["mean"], cfg["std"], cfg["res"], feat_dim)

In [ ]:
def make_transform(bb: Backbone):
    return transforms.Compose([
        transforms.Resize(bb.res), transforms.CenterCrop(bb.res),
        transforms.ToTensor(), transforms.Normalize(bb.mean, bb.std),
    ])

In [ ]:
@torch.no_grad()
def extract_features(loader, bb: Backbone, desc="Extracting features"):
    feats, labels = [], []
    for x, y in tqdm(loader, desc=desc, leave=False):
        feats.append(bb.forward(x.to(DEVICE)).float().cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

def syn_to_pixel(x):                                       
    return (x * _SRC_STD + _SRC_MEAN).clamp(0.0, 1.0)

@torch.no_grad()
def syn_features(imgs: torch.Tensor, bb: Backbone, bs=128):
    mean = torch.tensor(bb.mean, device=DEVICE).view(1, 3, 1, 1)
    std  = torch.tensor(bb.std,  device=DEVICE).view(1, 3, 1, 1)
    pix, out = syn_to_pixel(imgs), []
    for i in range(0, len(pix), bs):
        x = pix[i:i + bs].to(DEVICE)
        if x.shape[-1] != bb.res:
            x = F.interpolate(x, size=bb.res, mode="bicubic", align_corners=False).clamp(0, 1)
        out.append(bb.forward((x - mean) / std).float().cpu())
    return torch.cat(out)

def train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                       feat_dim=768, epochs=50, lr=1e-3, eval_every=10, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
    head = nn.Linear(feat_dim, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    g = torch.Generator()
    if seed is not None:
        g.manual_seed(seed)

    train_feat_loader = DataLoader(TensorDataset(train_feats, train_labels), batch_size=256, shuffle=True, generator=g)
    test_feat_loader = DataLoader(TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False)

    history: dict[int, dict[str, float]] = {}

    for epoch in tqdm(range(epochs), desc="Training"):
        head.train()
        total_loss, correct, total = 0.0, 0, 0

        for feats, y in train_feat_loader:
            feats, y = feats.to(DEVICE), y.to(DEVICE)
            logits = head(feats)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(y)

        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            head.eval()
            all_preds, all_labels_val = [], []
            with torch.no_grad():
                for feats, y in test_feat_loader:
                    all_preds.append(head(feats.to(DEVICE)).argmax(dim=1).cpu())
                    all_labels_val.append(y)
            all_preds = torch.cat(all_preds).numpy()
            all_labels_val = torch.cat(all_labels_val).numpy()

            history[epoch + 1] = {
                "loss": total_loss / total,
                "train_acc": correct / total,
                "f1_macro": f1_score(all_labels_val, all_preds, average="macro"),
                "f1_weighted": f1_score(all_labels_val, all_preds, average="weighted"),
            }

            m = history[epoch + 1]
            tqdm.write(
                f"Epoch {epoch+1:>3}/{epochs} | Loss: {m['loss']:.4f} | "
                f"Train Acc: {m['train_acc']*100:.1f}% | "
                f"F1 Macro: {m['f1_macro']*100:.1f}% | F1 Weighted: {m['f1_weighted']*100:.1f}%"
            )
    return head, history

In [ ]:
DISTILLED_BASE_DIR = "../logged_files/distillation/aqua20/dinov2_vitb"

class DistilledRun:
    def __init__(self, distilled_run: str):
        self.distilled_run = distilled_run
        data_path = Path(DISTILLED_BASE_DIR) / distilled_run / "data.pth"
        if not data_path.exists():
            raise FileNotFoundError(f"data.pth absent pour {distilled_run!r} (run incomplet ?)")
        self.syn_data: dict[str, torch.Tensor] = torch.load(data_path, map_location="cpu")
        self.total_time: float = 0.0
        self.head: nn.Linear | None = None
        self.history: dict | None = None

    def run_training(self, bb: Backbone, test_feats, test_labels,
                     epochs=50, lr=1e-3, eval_every=10, seed=None):
        start = time.time()
        feats = syn_features(self.syn_data["images"], bb)
        self.head, self.history = train_linear_probe(
            feats, self.syn_data["labels"], test_feats, test_labels,
            feat_dim=bb.feat_dim, epochs=epochs, lr=lr, eval_every=eval_every, seed=seed,
        )
        self.total_time = time.time() - start

## Setup multiple


In [ ]:
DISTILLED_BASE_DIR = "../logged_files/distillation/aqua20/dinov2_vitb"

In [ ]:
distilled_run_regex = re.compile(
    r"^distill_aqua20_h100"
    r"(?:_(physics_slurpp_frozen|physics_slurpp|physics_ppg|physics))?"
    r"_s\d+$"
)

distilled_runs: list[DistilledRun] = []
for p in sorted(Path(DISTILLED_BASE_DIR).iterdir()):
    if not (p.is_dir() and distilled_run_regex.match(p.name)):
        continue
    try:
        distilled_runs.append(DistilledRun(p.name))
    except FileNotFoundError as e:
        print(f"[skip] {e}")

print(f"{len(distilled_runs)} runs prêts")

120 runs prêts


In [ ]:
import numpy as np
import pandas as pd

def parse_run_name(name: str) -> tuple[int | None, str, int | None]:
    if name == "full_data":
        return None, "full", None
    m = re.match(
        r"^distill_aqua20_h100"
        r"(?:_(physics_slurpp_frozen|physics_slurpp|physics_ppg|physics))?"
        r"_s(\d+)$", name
    )
    if m is None:
        raise ValueError(f"run_name non reconnu : {name!r}")
    return None, (m.group(1) or "baseline"), int(m.group(2))

def final_metrics(history, last_k=3):
    epochs = sorted(history.keys())[-last_k:]
    return {
        "f1_macro":    float(np.mean([history[e]["f1_macro"]    for e in epochs])),
        "f1_weighted": float(np.mean([history[e]["f1_weighted"] for e in epochs])),
        "train_acc":   float(np.mean([history[e]["train_acc"]   for e in epochs])),
    }

In [ ]:
rows = []
for name in ARCHS:
    bb = load_backbone(name)
    tf = make_transform(bb)

    test_feats, test_labels = extract_features(
        DataLoader(datasets.ImageFolder(f"{DATA_ROOT}/test", transform=tf), batch_size=64, shuffle=False),
        bb, f"[{name}] test")

    # référence full-data (n=1) pour cette backbone
    full_feats, full_labels = extract_features(
        DataLoader(datasets.ImageFolder(f"{DATA_ROOT}/train", transform=tf), batch_size=64, shuffle=True),
        bb, f"[{name}] full train")
    _, hist_full = train_linear_probe(full_feats, full_labels, test_feats, test_labels,
                                      feat_dim=bb.feat_dim, epochs=50, seed=PROBE_SEED)
    mf = final_metrics(hist_full)
    rows.append({"arch": name, "IPC": None, "Variant": "full", "dseed": None,
                 "F1 macro (%)": mf["f1_macro"] * 100, "F1 weighted (%)": mf["f1_weighted"] * 100})
    del full_feats, full_labels

    for run in distilled_runs:
        ipc, variant, dseed = parse_run_name(run.distilled_run)
        run.run_training(bb, test_feats, test_labels, epochs=50, seed=PROBE_SEED)
        m = final_metrics(run.history)
        rows.append({"arch": name, "IPC": ipc, "Variant": variant, "dseed": dseed,
                     "F1 macro (%)": m["f1_macro"] * 100, "F1 weighted (%)": m["f1_weighted"] * 100})

    del bb, test_feats, test_labels
    gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(rows)

In [ ]:
summary = (df.groupby(["arch", "IPC", "Variant"], observed=True, dropna=False)
             .agg(macro_mean=("F1 macro (%)", "mean"), macro_std=("F1 macro (%)", "std"),
                  weighted_mean=("F1 weighted (%)", "mean"), weighted_std=("F1 weighted (%)", "std"),
                  n=("F1 macro (%)", "count"))
             .reset_index())
order = ["baseline", "physics", "physics_ppg", "physics_slurpp", "physics_slurpp_frozen", "full"]
summary["Variant"] = pd.Categorical(summary["Variant"], categories=order, ordered=True)
summary = summary.sort_values(["arch", "IPC", "Variant"], na_position="last").reset_index(drop=True)
fmt = lambda mu, sd: f"{mu:.2f}" if pd.isna(sd) else f"{mu:.2f} ± {sd:.2f}"
summary["F1 macro"]    = summary.apply(lambda r: fmt(r.macro_mean, r.macro_std), axis=1)
summary["F1 weighted"] = summary.apply(lambda r: fmt(r.weighted_mean, r.weighted_std), axis=1)
print(summary[["arch", "IPC", "Variant", "F1 macro", "F1 weighted", "n"]].to_string(index=False))